# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [62]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [63]:
# Task 1
# Treemap of fossil fuel TWh: Region -> Country -> Source
fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()
# compute top region/country for an insight title
region_tot = fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
country_tot = fossil.groupby('Country')['TWh'].sum().sort_values(ascending=False)
top_region = region_tot.index[0] if len(region_tot) else ''
top_country = country_tot.index[0] if len(country_tot) else ''
# CVD-safe discrete colours for fossil sources
colour_map = {'Coal': '#D55E00', 'Oil': '#CC79A7', 'Natural Gas': '#0072B2'}
fig1 = px.treemap(
    fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=colour_map,
)
# Grey parent nodes (Region and Country) by applying colours only to source-level labels
labels = list(fig1.data[0].labels)
colors = [colour_map.get(lbl, 'lightgrey') for lbl in labels]
fig1.data[0].marker.colors = colors
fig1.update_traces(texttemplate='%{label}<br>%{value:.0f} TWh', hovertemplate='%{label}<br>%{value:.0f} TWh')
fig1.update_layout(title_text=f'Most fossil-dependent region: {top_region} — country: {top_country}', margin=dict(t=60, l=25, r=25, b=25))
fig1.show()


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [64]:
# Task 2
# Sunburst showing total bill distribution: day -> time -> smoker
tips = px.data.tips()
agg = tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()
# CVD-safe blue/orange mapping for smoker status (Yes/No)
smoker_map = {'Yes': '#0072B2', 'No': '#D55E00'}
# build sunburst (values use summed total_bill)
fig2 = px.sunburst(agg, path=['day', 'time', 'smoker'], values='total_bill', color='smoker', color_discrete_map=smoker_map)
# grey out parent nodes and show percent-parent labels
labels = list(fig2.data[0].labels)
colors = [smoker_map.get(lbl, 'lightgrey') for lbl in labels]
fig2.data[0].marker.colors = colors
fig2.update_traces(texttemplate='%{label}<br>%{percentParent:.1%}', hovertemplate='%{label}<br>%{percentParent:.1%}')
# descriptive insight in title: where most spending happens
top = agg.loc[agg['total_bill'].idxmax()]
fig2.update_layout(title_text=f'Most spending: {top.day} {top.time} (smoker={top.smoker}) — {top.total_bill:.2f} total bill', margin=dict(t=60, l=25, r=25, b=25))
fig2.show()


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [65]:
# Task 3 — charts
# Compare low-carbon energy (Nuclear + Hydro) by country using treemap + horizontal bar
low = df.loc[df['Source_Type'] == 'Low-carbon'].copy()
country_tot = low.groupby('Country')['TWh'].sum().reset_index().sort_values('TWh', ascending=False)
# add dummy root for treemap
country_tot['All'] = 'Low-carbon'
fig_treemap = px.treemap(country_tot, path=['All', 'Country'], values='TWh', color='TWh', color_continuous_scale='Blues')
fig_treemap.update_traces(root_color='lightgrey', texttemplate='%{label}<br>%{value:.0f} TWh')
fig_treemap.update_layout(title_text='Low-carbon TWh by country (treemap)', margin=dict(t=60, l=25, r=25, b=25))
# horizontal bar, sorted by TWh
fig_bar = px.bar(country_tot, x='TWh', y='Country', orientation='h', color='TWh', color_continuous_scale='Blues')
fig_bar.update_traces(texttemplate='%{x:.0f} TWh', textposition='auto', hovertemplate='%{x:.0f} TWh')
# Ensure bars are ordered so the largest TWh are on top
fig_bar.update_layout(yaxis={'categoryorder':'total descending'})
lead = country_tot.iloc[0] if len(country_tot) else None
if lead is not None:
    fig_bar.update_layout(title_text=f"Leading low-carbon country: {lead['Country']} ({lead['TWh']:.0f} TWh)", margin=dict(t=60, l=150))
else:
    fig_bar.update_layout(margin=dict(t=60, l=150))
fig_treemap.show()
fig_bar.show()
